# ⚠️ For 12GB Colab: Use notebooks 01-06 instead
This single-file runner may exceed memory on free Colab.
Run in order: `01_setup_download` → `02_data_prep` → `03_phase1_baselines` → `04_phase2_embeddings` → `05_phase3_fewshot` → `06_visualization`

# EuroCropML x OLMoEarth Benchmark - Colab Runner

Runs the full benchmark pipeline on Google Colab with GPU.

Weights auto-download from HuggingFace. No file upload needed.

In [ ]:
# Cell 1: Setup
!git clone https://github.com/mahdikalantari555/eurocrop-olmoearth-benchmark.git
%cd eurocrop-olmoearth-benchmark
!apt-get install -y aria2
!pip install -r requirements.txt huggingface_hub -q
!pwd

In [ ]:
# Cell 2: Write cloud config
import yaml, os

print(f"CWD: {os.getcwd()}")

config = {
    'data': {
        'mode': 'local',
        'local_preprocess_dir': './preprocess',
        'local_split_dir': './split',
        'use_zenodo': True,
        'use_case': 'latvia_vs_estonia',
        'top_n_classes': 15,
        'test_split': 0.2,
        'random_seed': 42
    },
    'model': {
        'mode': 'cloud',
        'cloud_model_id': 'allenai/OlmoEarth-v1_1-Nano',
        'local_weights_path': './weights.pth',
        'batch_size': 32,
        'device': 'cuda'
    },
    'fewshot': {
        'shots': [5, 10, 20, 100, 200, 500],
        'n_repeats': 5
    },
    'output': {
        'metrics_dir': './results/metrics',
        'figures_dir': './results/figures'
    }
}

with open('config.yaml', 'w') as f:
    yaml.dump(config, f)

# Verify
with open('config.yaml') as f:
    print(f.read())

In [ ]:
# Cell 3: Download EuroCropML data from Zenodo (preprocess + split only)
import requests
import os

# Install aria2 if not available
!apt-get install -y aria2 -q

record_id = 15095445
record = requests.get(f"https://zenodo.org/api/records/{record_id}").json()

files_needed = ["preprocess.zip", "split.zip"]

for file in record["files"]:
    if file["key"] not in files_needed:
        continue
    filename = file["key"]
    filesize = file["size"]
    url = file["links"]["self"]
    zip_path = filename

    if os.path.exists(zip_path) and os.path.getsize(zip_path) == filesize:
        print(f"Already downloaded: {filename}")
    else:
        print(f"Downloading: {filename} ({filesize / 1e6:.1f} MB)")
        !aria2c -x 16 -s 16 -o "{filename}" "{url}"

    if os.path.exists(zip_path):
        print(f"Extracting: {filename}")
        !unzip -q -o "{zip_path}"
        os.remove(zip_path)
    else:
        print(f"Warning: {zip_path} not found after download")

print("Done!")
!ls -d preprocess/ split/

In [ ]:
# Cell 4: Phase 1 - Classical baselines
%run experiments/phase1_baselines.py

In [ ]:
# Cell 5: Phase 2 - OLMoEarth embeddings (weights download from HF automatically)
%run experiments/phase2_embeddings.py

In [ ]:
# Cell 6: Phase 3 - Few-shot comparison
%run experiments/phase3_fewshot.py

In [ ]:
# Cell 7: Visualizations
import json
import matplotlib.pyplot as plt
import os
import yaml

if os.path.exists('results/metrics/phase3_fewshot.json'):
    from src.viz.fewshot_curve import plot_fewshot_curve
    with open('results/metrics/phase3_fewshot.json') as f:
        results = json.load(f)
    plot_fewshot_curve(results, 'results/figures/fewshot_curve.png')
    plt.show()

if os.path.exists('results/metrics/emb_train.npy'):
    import numpy as np
    from src.viz.umap_viz import plot_umap
    from src.data.loader import load_split_zenodo
    with open('config.yaml') as f:
        cfg = yaml.safe_load(f)
    emb = np.load('results/metrics/emb_train.npy')
    splits = load_split_zenodo(
        cfg['data']['local_preprocess_dir'],
        cfg['data']['local_split_dir'],
        cfg['data']['use_case'], 'all')
    _, y, _ = splits['train']
    y = np.array(y)
    plot_umap(emb[:len(y)], y, save_path='results/figures/umap_embeddings.png')
    plt.show()

In [ ]:
# Cell 7.5: Print summary of all results
import json
import glob

print("=" * 70)
print("BENCHMARK RESULTS SUMMARY")
print("=" * 70)

for phase_name in ["phase1", "phase2", "phase3", "phase4"]:
    files = sorted(glob.glob(f'results/metrics/{phase_name}*.json'))
    if not files:
        continue
    
    print(f"\n{'─' * 50}")
    print(f"  {phase_name.upper().replace('_', ' ')}")
    print(f"{'─' * 50}")
    
    for f in files:
        with open(f) as fh:
            data = json.load(fh)
        
        name = f.split('/')[-1].replace('.json', '')
        
        if isinstance(data, dict) and 'overall_accuracy' in data:
            print(f"  {name}:")
            print(f"    OA={data['overall_accuracy']:.3f} | F1={data['macro_f1']:.3f} | "
                  f"Kappa={data['kappa']:.3f}")
        elif isinstance(data, dict) and 'rf_f1' in data:
            print(f"  {name}:")
            for shot, metrics in sorted(data.items(), key=lambda x: int(x[0])):
                print(f"    {shot}-shot: RF={metrics['rf_f1']:.3f} | "
                      f"LGBM={metrics['lgbm_f1']:.3f} | OLMO={metrics['olmo_lgbm_f1']:.3f}")

print("\n" + "=" * 70)

In [ ]:
# Cell 8: Download all results (metrics + logs + figures)
from google.colab import files
import glob
import os

print("Downloading metrics JSON files...")
for f in sorted(glob.glob('results/metrics/*.json')):
    print(f"  {f}")
    files.download(f)

print("\nDownloading log files...")
for f in sorted(glob.glob('results/metrics/*.log') + glob.glob('results/metrics/*_log.txt')):
    print(f"  {f}")
    files.download(f)

print("\nDownloading figures...")
for f in sorted(glob.glob('results/figures/*.png')):
    print(f"  {f}")
    files.download(f)

print("\nAll results downloaded!")